# Geographic Analysis

**InsightForge Business Analytics Studio — Python Analyst Toolkit**

## Purpose
Break down revenue and customer counts by `region` and `country` to see where the business is strongest and where it might be underinvested.

## Methodology
- Aggregate `revenue` and distinct `customerId` counts by `region`, then by `country`.
- Rank both dimensions by total revenue.
- Visualize with simple `matplotlib` bar charts -- **no paid map APIs or external tile services** are used anywhere in this toolkit, by design (no network calls, no API keys required).

## Assumptions
- `region`/`country` on each order reflect the customer's home region/country as recorded at order time.
- Revenue is not currency-adjusted; the synthetic dataset uses a single currency throughout.

## Limitations
- This is bar-chart-level geographic analysis, not a mapped/spatial visualization -- if you need an actual map, plug the `latitude`/`longitude` columns (present in `customers.csv`) into a mapping library of your choice outside this toolkit.
- Regional boundaries here are coarse (4 regions, ~10 countries in the sample data); your own export may have a different geographic footprint.
- Latitude/longitude in the synthetic sample data are randomly jittered around each country's rough centroid -- they approximate a plausible spread, not real customer addresses.

In [1]:
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

TOOLKIT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, TOOLKIT_DIR)
from clean_data import load_csv, clean_dataframe

OUTPUT_DIR = os.path.join(TOOLKIT_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

orders_path = os.path.join(TOOLKIT_DIR, "sample_data", "orders.csv")
raw_orders = load_csv(orders_path)
orders, cleaning_summary = clean_dataframe(raw_orders)
print(cleaning_summary)

Data Cleaning Summary

- Removed 11 exact duplicate row(s) (kept first occurrence).
- No invalid negative values found in quantity/revenue/cost/discount/rating.
- Missing values detected (left as-is; not auto-filled to avoid inventing data):
    - 'campaign': 3831 missing (60.6% of rows)
    - 'region': 6 missing (0.1% of rows)
    - 'marketingSource': 4 missing (0.1% of rows)
    - 'discount': 3 missing (0.0% of rows)
    - 'channel': 2 missing (0.0% of rows)
- Statistical outliers detected via IQR method (reported only, not removed):
    - 'quantity': 201 value(s) outside [-0.5, 3.5]
    - 'revenue': 580 value(s) outside [-39.06, 185.38]
    - 'cost': 507 value(s) outside [-20.03, 94.18]
    - 'profit': 442 value(s) outside [-30.52, 105.58]
    - 'discount': 615 value(s) outside [-0.08, 0.12]
  Outliers are left in the cleaned data by default -- in real business data these are often legitimate large orders, not errors. Review manually before excluding.

Rows before cleaning: 6326
Row

## Revenue and Customers by Region

In [2]:
if not {"region", "revenue", "customerId"}.issubset(orders.columns):
    print("Unavailable: 'region', 'revenue', or 'customerId' column not present.")
    region_summary = None
else:
    region_summary = orders.groupby("region").agg(
        totalRevenue=("revenue", "sum"),
        orders=("orderId", "count"),
        customers=("customerId", "nunique"),
    ).sort_values("totalRevenue", ascending=False)
    region_summary.to_csv(os.path.join(OUTPUT_DIR, "05_region_summary.csv"))
    print("Saved output/05_region_summary.csv")
    display(region_summary)

Saved output/05_region_summary.csv


,totalRevenue,orders,customers
region,,,
North America,152683.71,1640,520
Latin America,150013.04,1550,535
Europe,148431.70,1568,512
Asia Pacific,147212.68,1551,516


In [3]:
if region_summary is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    region_summary["totalRevenue"].sort_values().plot(kind="barh", ax=axes[0], color="#2a5599")
    axes[0].set_title("Revenue by Region")
    axes[0].set_xlabel("Revenue ($)")

    region_summary["customers"].sort_values().plot(kind="barh", ax=axes[1], color="#3c9c6a")
    axes[1].set_title("Distinct Customers by Region")
    axes[1].set_xlabel("Customers")

    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "05_revenue_by_region.png"), dpi=150)
    plt.close(fig)
    print("Saved output/05_revenue_by_region.png")

Saved output/05_revenue_by_region.png


## Revenue Ranking by Country

In [4]:
if not {"country", "revenue"}.issubset(orders.columns):
    print("Unavailable: 'country' or 'revenue' column not present.")
    country_summary = None
else:
    country_summary = orders.groupby("country").agg(
        totalRevenue=("revenue", "sum"),
        orders=("orderId", "count"),
        customers=("customerId", "nunique"),
    ).sort_values("totalRevenue", ascending=False)
    country_summary["rank"] = range(1, len(country_summary) + 1)
    country_summary.to_csv(os.path.join(OUTPUT_DIR, "05_country_ranking.csv"))
    print("Saved output/05_country_ranking.csv")
    display(country_summary)

Saved output/05_country_ranking.csv


,totalRevenue,orders,customers,rank
country,,,,
Mexico,86008.07,806,272,1
Canada,76422.48,857,274,2
United States,76370.28,785,246,3
Brazil,64188.29,746,263,4
Singapore,52054.43,535,169,5
France,51819.40,548,181,6
Germany,50105.41,556,165,7
Japan,48891.63,533,187,8
United Kingdom,46532.17,465,166,9


In [5]:
if country_summary is not None:
    fig, ax = plt.subplots(figsize=(9, 6))
    country_summary["totalRevenue"].sort_values().plot(kind="barh", ax=ax, color="#c96a2e")
    ax.set_title("Revenue by Country")
    ax.set_xlabel("Revenue ($)")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "05_revenue_by_country.png"), dpi=150)
    plt.close(fig)
    print("Saved output/05_revenue_by_country.png")

Saved output/05_revenue_by_country.png


## Summary
This notebook wrote `05_region_summary.csv`, `05_revenue_by_region.png`, `05_country_ranking.csv`, and `05_revenue_by_country.png` to `output/`.

No map APIs or external services were used -- all charts are static `matplotlib` bar charts generated entirely offline.